# Simple RAG-LLM Action Planning

This notebook loads predictions, searches knowledge base using contributing features, and generates action plans using LLM.

In [12]:
# All Imports
import os
import json
import re
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, List, Tuple
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from openai import OpenAI

load_dotenv()

True

In [13]:
# 1. Setup and Load Data
# Paths
rag_predictions_dir = Path("RAG_docs/predictions")
rag_knowledge_file = Path("RAG_docs/knowledge/ddos-synthetic-dataset.json")
results_action_dir = Path("RAG_docs/action_plans")
os.makedirs(results_action_dir, exist_ok=True)

# Load all prediction files from the directory
prediction_files = list(rag_predictions_dir.glob("*.json"))
if not prediction_files:
    raise FileNotFoundError(f"No JSON files found in {rag_predictions_dir}")

print(f"Found {len(prediction_files)} prediction file(s) in {rag_predictions_dir}")

# Load and combine all predictions from all files
predictions_data = []
for prediction_file in prediction_files:
    print(f"  Loading: {prediction_file.name}")
    with open(prediction_file, "r", encoding="utf-8") as f:
        file_data = json.load(f)
    
    # Handle single object or array
    if isinstance(file_data, dict):
        file_data = [file_data]
    elif isinstance(file_data, list):
        pass  # Already a list
    else:
        print(f"    Warning: Unexpected data type in {prediction_file.name}, skipping")
        continue
    
    # Add source file info to each prediction
    for sample in file_data:
        if isinstance(sample, dict):
            sample["_source_file"] = prediction_file.name
    
    predictions_data.extend(file_data)
    print(f"    Loaded {len(file_data)} prediction(s) from {prediction_file.name}")

print(f"\nTotal: Loaded {len(predictions_data)} prediction(s) from {len(prediction_files)} file(s)")

# Load knowledge base
with open(rag_knowledge_file, "r", encoding="utf-8") as f:
    knowledge_base = json.load(f)
print(f"Loaded {len(knowledge_base)} documents from knowledge base")


Found 1 prediction file(s) in RAG_docs\predictions
  Loading: sample_predictions_detailed_20260207_111609.json
    Loaded 2 prediction(s) from sample_predictions_detailed_20260207_111609.json

Total: Loaded 2 prediction(s) from 1 file(s)
Loaded 121 documents from knowledge base


In [14]:
# 2. Create Vector Store (Offline - No API Key Required)
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

# Process knowledge base into documents
rag_documents = []
for idx, item in enumerate(knowledge_base):
    title = item.get("title", "")
    text = item.get("text", "")
    doc_text = f"Title: {title}\n\nContent: {text}"
    
    chunks = text_splitter.split_text(doc_text)
    for chunk_idx, chunk in enumerate(chunks):
        rag_documents.append(Document(
            page_content=chunk,
            metadata={"title": title, "text": text, "item_index": idx, "chunk_index": chunk_idx}
        ))

print(f"Created {len(rag_documents)} document chunks")

# Create vector store using local embeddings (offline)
print("Creating vector embeddings using local model (this may take a few minutes)...")
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(rag_documents, embeddings)
print("✓ Vector store created (offline, no API key required)")


Created 441 document chunks
Creating vector embeddings using local model (this may take a few minutes)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Vector store created (offline, no API key required)


In [15]:
# 4a. Extract Party Features (Reusable Function)
def get_party_features(feature_contributions, party_name):
    """Extract features with contribution > 0 for a party.
    
    Args:
        feature_contributions: Dictionary of feature contributions from SHAP explanation
        party_name: Name of the party (e.g., "evidence_volume_rate_agent1")
    
    Returns:
        List of feature dictionaries with name, pct_contribution, and abs_shap_value
    """
    party_features = feature_contributions.get(party_name, {})
    if not isinstance(party_features, dict):
        return []
    
    features_list = []
    for feat_name, feat_data in party_features.items():
        if not isinstance(feat_data, dict):
            continue
        pct_contrib = feat_data.get("pct_contribution", 0.0)
        abs_shap = feat_data.get("abs_shap_value", 0.0)
        if pct_contrib > 0 or abs_shap > 0:
            features_list.append({
                "name": feat_name,
                "pct_contribution": pct_contrib,
                "abs_shap_value": abs_shap
            })
    
    # Sort by absolute SHAP value (descending)
    features_list.sort(key=lambda x: x["abs_shap_value"], reverse=True)
    return features_list

def build_features_context(party1_features, party2_features, party3_features, top_n=10):
    """Build formatted context string for contributing features by party.
    
    Args:
        party1_features: List of feature dicts for Party1 (evidence_volume_rate)
        party2_features: List of feature dicts for Party2 (evidence_packet_size)
        party3_features: List of feature dicts for Party3 (evidence_timing_direction)
        top_n: Number of top features to include per party (default: 10)
    
    Returns:
        Formatted string with contributing features, or empty string if no features
    """
    if not (party1_features or party2_features or party3_features):
        return ""
    
    features_context = "\n\nContributing Features (by party, contribution > 0):\n"
    
    if party1_features:
        features_context += f"\nParty1 (evidence_volume_rate):\n"
        for feat in party1_features[:top_n]:
            features_context += f"  - {feat['name']}: {feat['pct_contribution']:.2%} contribution (abs_shap: {feat['abs_shap_value']:.4f})\n"
    
    if party2_features:
        features_context += f"\nParty2 (evidence_packet_size):\n"
        for feat in party2_features[:top_n]:
            features_context += f"  - {feat['name']}: {feat['pct_contribution']:.2%} contribution (abs_shap: {feat['abs_shap_value']:.4f})\n"
    
    if party3_features:
        features_context += f"\nParty3 (evidence_timing_direction):\n"
        for feat in party3_features[:top_n]:
            features_context += f"  - {feat['name']}: {feat['pct_contribution']:.2%} contribution (abs_shap: {feat['abs_shap_value']:.4f})\n"
    
    return features_context

In [16]:
# 4. Smart RAG Query Generation (Feature-name -> Human Signal Rules)
# Feature-name -> human signal rules
SIGNAL_RULES: List[Tuple[re.Pattern, str]] = [
    # DNS / UDP
    (re.compile(r"dns", re.I), "abnormal DNS activity (possible amplification/fan-in)"),
    (re.compile(r"\budps?\b", re.I), "elevated UDP traffic patterns"),
    (re.compile(r"dns_port(_src)?_count", re.I), "DNS source fan-in / query-source spike"),

    # TCP flags
    (re.compile(r"\brst(_packets)?\b", re.I), "elevated TCP RST responses (reset bursts)"),
    (re.compile(r"\bsyn(_packets)?\b", re.I), "SYN surge behavior (flood-like pattern)"),
    (re.compile(r"\back(_packets)?\b", re.I), "abnormal ACK behavior"),
    (re.compile(r"\bpsh(_packets)?\b", re.I), "PSH-flag anomalies"),

    # Timing
    (re.compile(r"\bpiat\b", re.I), "inter-arrival timing irregularities"),
    (re.compile(r"duration", re.I), "flow duration anomalies"),

    # Volume / size
    (re.compile(r"bytes", re.I), "unusual byte volume distribution"),
    (re.compile(r"packet_count|\bpackets\b", re.I), "packet-rate / packet-count anomalies"),
    (re.compile(r"_mean_ps\b|packet_size", re.I), "packet-size shift (mean/variance change)"),

    # Ports / scan-like
    (re.compile(r"unique_ports|vul_ports|http_ports", re.I), "suspicious port/service targeting patterns"),
]

def extract_top_features(sample: Dict[str, Any], top_n: int = 7) -> List[str]:
    """Top feature names by abs_shap_value across all parties (non-zero)."""
    feat_contribs = (sample.get("shap_explanation", {}) or {}).get("feature_contributions", {}) or {}

    scored: List[Tuple[float, str]] = []
    for _, feats in feat_contribs.items():
        if not isinstance(feats, dict):
            continue
        for feat_name, v in feats.items():
            if not isinstance(v, dict):
                continue
            abs_val = float(v.get("abs_shap_value", 0.0) or 0.0)
            if abs_val > 0:
                scored.append((abs_val, feat_name))

    scored.sort(reverse=True, key=lambda x: x[0])

    seen = set()
    top_feats = []
    for _, f in scored:
        if f not in seen:
            seen.add(f)
            top_feats.append(f)
        if len(top_feats) >= top_n:
            break
    return top_feats

def features_to_signals(feature_names: List[str], max_signals: int = 3) -> List[str]:
    """Map feature names -> deduped human signals."""
    signals: List[str] = []
    for f in feature_names:
        for pattern, signal in SIGNAL_RULES:
            if pattern.search(f):
                if signal not in signals:
                    signals.append(signal)
                break  # only first match per feature
        if len(signals) >= max_signals:
            break
    return signals

def build_dynamic_rag_query(sample: Dict[str, Any],
                            intent: str = "Explain and mitigate",
                            max_signals: int = 3) -> str:
    """Return a single-sentence dynamic RAG query."""
    label = sample.get("predicted_label") or sample.get("true_label") or "UNKNOWN"
    top_feats = extract_top_features(sample, top_n=10)
    signals = features_to_signals(top_feats, max_signals=max_signals)

    if signals:
        if len(signals) == 1:
            signals_text = signals[0]
        else:
            signals_text = ", ".join(signals[:-1]) + f", and {signals[-1]}"
        return f"{intent} a {label} attack characterized by {signals_text}."
    else:
        return f"{intent} a {label} attack and provide SOC triage steps and mitigations."

In [17]:
# 5. Create LLM Prompt with Party Evidence and RAG Context
def create_prompt(sample_data, rag_results):
    """Create prompt with prediction details, party evidence, contributing features, and RAG context (top 3 docs)."""
    sample_id = sample_data.get("sample_id", 0)
    predicted_label = sample_data.get("predicted_label", "UNKNOWN")
    confidence = sample_data.get("confidence", 0.0)
    
    # Extract party contributions
    shap_explanation = sample_data.get("shap_explanation", {})
    party_contributions_pct = shap_explanation.get("party_contributions_pct", {})
    feature_contributions = shap_explanation.get("feature_contributions", {})
    dominant_party = shap_explanation.get("dominant_party", "unknown")
    
    # Get party percentages (default to 0 if not found)
    p1_pct = party_contributions_pct.get("evidence_volume_rate_agent1", 0.0)
    p2_pct = party_contributions_pct.get("evidence_packet_size_agent2", 0.0)
    p3_pct = party_contributions_pct.get("evidence_timing_direction_agent3", 0.0)
    
    # Map dominant party to readable name
    dominant_mapping = {
        "evidence_volume_rate_agent1": "evidence_volume_rate (Party1)",
        "evidence_packet_size_agent2": "evidence_packet_size (Party2)",
        "evidence_timing_direction_agent3": "evidence_timing_direction (Party3)"
    }
    dominant_evidence_name = dominant_mapping.get(dominant_party, dominant_party)
    
    # Get features for each party using reusable function from top
    party1_features = get_party_features(feature_contributions, "evidence_volume_rate_agent1")
    party2_features = get_party_features(feature_contributions, "evidence_packet_size_agent2")
    party3_features = get_party_features(feature_contributions, "evidence_timing_direction_agent3")
    
    # Build contributing features context using reusable function
    features_context = build_features_context(party1_features, party2_features, party3_features, top_n=10)
    
    # Use top 3 RAG results
    top_3_results = rag_results[:3]
    
    # Build RAG context
    rag_context = ""
    if top_3_results:
        rag_context = "\n\nKnowledge Base Context:\n"
        for idx, result in enumerate(top_3_results, 1):
            rag_context += f"\n[{idx}] {result['title']}\n"
            rag_context += f"{result['text'][:1000]}\n"
    else:
        rag_context = "\n\nKnowledge Base: No relevant documents found."
    
    prompt = f"""You are a cybersecurity expert analyzing attack predictions and generating action plans.
The goal is to support agent decisions on WHICH evidence party should trigger actions
and WHAT actions are appropriate based on domain knowledge.

Prediction summary:
- sample_id: {sample_id}
- predicted_label: {predicted_label}
- confidence: {confidence:.1%}

Explainability (party-level evidence):
- evidence_volume_rate (Party1): {p1_pct:.2%}
- evidence_packet_size (Party2): {p2_pct:.2%}
- evidence_timing_direction (Party3): {p3_pct:.2%}
- dominant_evidence: {dominant_evidence_name}
{features_context}

{rag_context}

Task:
Based on the prediction details and knowledge base content above, provide an action plan that:
1) Understands how {predicted_label} manifests in each evidence type
   (volume/rate, packet size, timing/direction).
2) Decides which evidence party should trigger an agent action first.
3) Determines what agent actions are appropriate for each evidence party
   (e.g., rate limiting, filtering, flow inspection, escalation).
4) Validates whether non-dominant parties provide supporting or contradictory signals.
5) Supports confidence-aware decisions (given confidence = {confidence:.1%}).
6) CRITICAL: Each action MUST specify which party (Party1, Party2, or Party3) will execute it.

Output:
Return a JSON object with the following structure:
{{
  "threat_level": "Critical|High|Medium|Low",
  "primary_actions": [
    {{
      "action": "description of action",
      "party": "Party1|Party2|Party3",
      "party_evidence_type": "evidence_volume_rate|evidence_packet_size|evidence_timing_direction",
      "rationale": "why this party should execute this action"
    }},
    ...
  ],
  "supporting_actions": [
    {{
      "action": "description of action",
      "party": "Party1|Party2|Party3",
      "party_evidence_type": "evidence_volume_rate|evidence_packet_size|evidence_timing_direction",
      "rationale": "why this party should execute this action"
    }},
    ...
  ],
  "reasoning": "Brief explanation of the decision and party assignments",
  "execution_priority": "Immediate|High|Standard|Low",
  "knowledge_sources_used": ["source1", "source2", ...]
}}

IMPORTANT: 
- Each action in primary_actions and supporting_actions must be an object with "action", "party", "party_evidence_type", and "rationale" fields.
- The "party" field must be exactly "Party1", "Party2", or "Party3".
- The "party_evidence_type" must match: "evidence_volume_rate" for Party1, "evidence_packet_size" for Party2, "evidence_timing_direction" for Party3.
- Actions should be assigned to parties based on which evidence type is most relevant to that action."""
    
    return prompt

In [18]:
def create_prompt_without_RAG(sample_data):
    """Create prompt with prediction details, party evidence, contributing features, WITHOUT RAG context."""
    sample_id = sample_data.get("sample_id", 0)
    predicted_label = sample_data.get("predicted_label", "UNKNOWN")
    confidence = sample_data.get("confidence", 0.0)
    
    # Extract party contributions
    shap_explanation = sample_data.get("shap_explanation", {})
    party_contributions_pct = shap_explanation.get("party_contributions_pct", {})
    feature_contributions = shap_explanation.get("feature_contributions", {})
    
    # Get party percentages (default to 0 if not found)
    p1_pct = party_contributions_pct.get("evidence_volume_rate_agent1", 0.0)
    p2_pct = party_contributions_pct.get("evidence_packet_size_agent2", 0.0)
    p3_pct = party_contributions_pct.get("evidence_timing_direction_agent3", 0.0)
    
    # Get features for each party using reusable function from top
    party1_features = get_party_features(feature_contributions, "evidence_volume_rate_agent1")
    party2_features = get_party_features(feature_contributions, "evidence_packet_size_agent2")
    party3_features = get_party_features(feature_contributions, "evidence_timing_direction_agent3")
    
    # Build contributing features context using reusable function
    features_context = build_features_context(party1_features, party2_features, party3_features, top_n=10)
    
    prompt = f"""You are a cybersecurity expert analyzing attack predictions and generating action plans.
The goal is to support agent decisions on WHICH evidence party should trigger actions
and WHAT actions are appropriate based on domain knowledge.

Prediction summary:
- sample_id: {sample_id}
- predicted_label: {predicted_label}
- confidence: {confidence:.1%}

Explainability (party-level evidence):
- evidence_volume_rate (Party1): {p1_pct:.2%}
- evidence_packet_size (Party2): {p2_pct:.2%}
- evidence_timing_direction (Party3): {p3_pct:.2%}
{features_context}

Task:
Based on the prediction details above, provide an action plan that:
1) Understands how {predicted_label} manifests in each evidence type
   (volume/rate, packet size, timing/direction).
2) Decides which evidence party should trigger an agent action first.
3) Determines what agent actions are appropriate for each evidence party
   (e.g., rate limiting, filtering, flow inspection, escalation).
4) Validates whether non-dominant parties provide supporting or contradictory signals.
5) Supports confidence-aware decisions (given confidence = {confidence:.1%}).
6) CRITICAL: Each action MUST specify which party (Party1, Party2, or Party3) will execute it.

Output:
Return a JSON object with the following structure:
{{
  "threat_level": "Critical|High|Medium|Low",
  "primary_actions": [
    {{
      "action": "description of action",
      "party": "Party1|Party2|Party3",
      "party_evidence_type": "evidence_volume_rate|evidence_packet_size|evidence_timing_direction",
      "rationale": "why this party should execute this action"
    }},
    ...
  ],
  "supporting_actions": [
    {{
      "action": "description of action",
      "party": "Party1|Party2|Party3",
      "party_evidence_type": "evidence_volume_rate|evidence_packet_size|evidence_timing_direction",
      "rationale": "why this party should execute this action"
    }},
    ...
  ],
  "reasoning": "Brief explanation of the decision and party assignments",
  "execution_priority": "Immediate|High|Standard|Low",
  "knowledge_sources_used": ["source1", "source2", ...]
}}

IMPORTANT: 
- Each action in primary_actions and supporting_actions must be an object with "action", "party", "party_evidence_type", and "rationale" fields.
- The "party" field must be exactly "Party1", "Party2", or "Party3".
- The "party_evidence_type" must match: "evidence_volume_rate" for Party1, "evidence_packet_size" for Party2, "evidence_timing_direction" for Party3.
- Actions should be assigned to parties based on which evidence type is most relevant to that action."""
    
    return prompt

In [19]:
# 3. RAG Search Function
def search_rag_knowledge(query, top_k=5):
    """Search knowledge base and return documents with title and text."""
    if vector_store is None:
        return []
    
    results = vector_store.similarity_search_with_score(query, k=top_k)
    
    formatted_results = []
    seen_titles = set()
    
    for doc, score in results:
        title = doc.metadata.get("title", "Unknown")
        text = doc.metadata.get("text", "")
        
        # Avoid duplicates - use first occurrence of each title
        if title not in seen_titles:
            seen_titles.add(title)
            similarity_score = 1 / (1 + score) if score > 0 else 1.0
            formatted_results.append({
                "title": title,
                "text": text,
                "score": similarity_score
            })
    
    return formatted_results

In [20]:
# 8. Save Comparison File Function
def save_comparison_file(sample, query, rag_results, llm_response_with_rag, llm_response_without_rag, 
                         results_action_dir, with_rag_file, without_rag_file):
    """Save a comparison JSON file containing both with-RAG and without-RAG action plans.
    
    Args:
        sample: Prediction sample data
        query: RAG query used
        rag_results: RAG search results
        llm_response_with_rag: LLM response with RAG context
        llm_response_without_rag: LLM response without RAG context
        results_action_dir: Directory to save results
        with_rag_file: Path to the with-RAG action plan file
        without_rag_file: Path to the without-RAG action plan file
    """
    comparison = {
        "sample_id": sample.get("sample_id"),
        "prediction": {
            "predicted_label": sample.get("predicted_label"),
            "confidence": float(sample.get("confidence", 0.0))
        },
        "comparison_metadata": {
            "compared_at": datetime.now().isoformat(),
            "with_rag_file": with_rag_file.name if with_rag_file else None,
            "without_rag_file": without_rag_file.name if without_rag_file else None,
            "rag_query": query,
            "rag_documents_found": len(rag_results)
        },
        "rag_context": {
            "documents_used": len(rag_results),
            "top_documents": [
                {
                    "title": r["title"],
                    "similarity_score": float(r["score"]),
                    "text_preview": r["text"][:200] + "..." if len(r["text"]) > 200 else r["text"]
                }
                for r in rag_results[:3]
            ]
        },
        "action_plans": {
            "with_rag": {
                "threat_level": llm_response_with_rag.get("threat_level") if llm_response_with_rag else None,
                "execution_priority": llm_response_with_rag.get("execution_priority") if llm_response_with_rag else None,
                "primary_actions": llm_response_with_rag.get("primary_actions", []) if llm_response_with_rag else [],
                "supporting_actions": llm_response_with_rag.get("supporting_actions", []) if llm_response_with_rag else [],
                "reasoning": llm_response_with_rag.get("reasoning", "") if llm_response_with_rag else "",
                "knowledge_sources_used": llm_response_with_rag.get("knowledge_sources_used", []) if llm_response_with_rag else [],
            },
            "without_rag": {
                "threat_level": llm_response_without_rag.get("threat_level") if llm_response_without_rag else None,
                "execution_priority": llm_response_without_rag.get("execution_priority") if llm_response_without_rag else None,
                "primary_actions": llm_response_without_rag.get("primary_actions", []) if llm_response_without_rag else [],
                "supporting_actions": llm_response_without_rag.get("supporting_actions", []) if llm_response_without_rag else [],
                "reasoning": llm_response_without_rag.get("reasoning", "") if llm_response_without_rag else "",
                "knowledge_sources_used": llm_response_without_rag.get("knowledge_sources_used", []) if llm_response_without_rag else [],
            }
        },
        "differences": {
            "threat_level_different": (llm_response_with_rag.get("threat_level") if llm_response_with_rag else None) != 
                                      (llm_response_without_rag.get("threat_level") if llm_response_without_rag else None),
            "priority_different": (llm_response_with_rag.get("execution_priority") if llm_response_with_rag else None) != 
                                 (llm_response_without_rag.get("execution_priority") if llm_response_without_rag else None),
            "primary_actions_different": (llm_response_with_rag.get("primary_actions", []) if llm_response_with_rag else []) != 
                                        (llm_response_without_rag.get("primary_actions", []) if llm_response_without_rag else []),
            "num_primary_actions_with_rag": len(llm_response_with_rag.get("primary_actions", [])) if llm_response_with_rag else 0,
            "num_primary_actions_without_rag": len(llm_response_without_rag.get("primary_actions", [])) if llm_response_without_rag else 0
        }
    }
    
    # Convert any numpy types to native Python types
    comparison = convert_to_json_serializable(comparison)
    
    # Create filename with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    comparison_file = results_action_dir / f"comparison_sample_{sample.get('sample_id')}_{timestamp}.json"
    
    with open(comparison_file, "w", encoding="utf-8") as f:
        json.dump(comparison, f, indent=2, ensure_ascii=False)
    
    return comparison_file

In [21]:
# 7. Save Action Plan
def convert_to_json_serializable(obj):
    """Convert numpy types and other non-serializable types to JSON-compatible types."""
    import numpy as np
    
    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_to_json_serializable(value) for key, value in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_json_serializable(item) for item in obj]
    else:
        return obj

def save_action_plan(sample, query, rag_results, llm_response, results_action_dir):
    """Save action plan in the specified format."""
    result = {
        "sample_id": sample.get("sample_id"),
        "prediction": {
            "predicted_label": sample.get("predicted_label"),
            "confidence": float(sample.get("confidence", 0.0))
        },
        "rag_info": {
            "documents_used": len(rag_results),
            "search_method": "vector_similarity",
            "search_queries": [query] if query else [],
            "top_results": [
                {
                    "title": r["title"],
                    "text": r["text"],
                    "similarity_score": float(r["score"]),
                    "source_file": rag_knowledge_file.name
                }
                for r in rag_results[:3]
            ]
        },
        "llm_action_plan": llm_response,
        "processed_at": datetime.now().isoformat()
    }
    
    # Convert any numpy types to native Python types
    result = convert_to_json_serializable(result)
    
    # Create filename with timestamp (include microseconds to avoid collisions)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    output_file = results_action_dir / f"action_plan_sample_{sample.get('sample_id')}_{timestamp}.json"
    
    try:
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        # Verify file was created
        if not output_file.exists():
            raise FileNotFoundError(f"File was not created: {output_file}")
        return output_file
    except Exception as e:
        print(f"Error in save_action_plan: {e}")
        raise

In [22]:
# 6. Call LLM API
def call_llm_api(prompt):
    """Call LLM API and return JSON response."""
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("ERROR: OPENAI_API_KEY not set")
        return None
    
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "You are a cybersecurity expert. Return only valid JSON."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )
    
    response_text = response.choices[0].message.content.strip()
    try:
        start = response_text.find('{')
        end = response_text.rfind('}') + 1
        if start >= 0 and end > start:
            return json.loads(response_text[start:end])
    except Exception as e:
        print(f"Warning: Could not parse JSON: {e}")
    
    return {
        "threat_level": "Unknown",
        "primary_actions": ["Unable to parse response"],
        "supporting_actions": [],
        "reasoning": response_text[:200],
        "execution_priority": "Standard",
        "knowledge_sources_used": []
    }

# 7. Process Predictions
# Process ALL predictions (including BENIGN) with RAG and attack-focused prompts
for sample in predictions_data:
    print("="*80)
    print(f"Processing Sample ID: {sample.get('sample_id')}")
    predicted_label = sample.get('predicted_label', 'UNKNOWN')
    print(f"Predicted Label: {predicted_label}")
    print(f"Confidence: {sample.get('confidence', 0):.1%}")
    
    # Process ALL predictions with RAG (including BENIGN)
    # Create smart RAG query from contributing features
    query = build_dynamic_rag_query(sample)
    print(f"\nRAG Query: {query}")
    
    # Search knowledge base (get top 5, use top 3)
    rag_results = search_rag_knowledge(query, top_k=5)
    print(f"Found {len(rag_results)} relevant documents")
    
    if rag_results:
        print("\nTop 3 documents for LLM context:")
        for idx, result in enumerate(rag_results[:3], 1):
            print(f"  {idx}. {result['title']} (Score: {result['score']:.2%})")
    
    # Create prompt with top 3 RAG results (for all predictions including BENIGN)
    prompt = create_prompt(sample, rag_results)
    prompt_without_rag = create_prompt_without_RAG(sample)
    
    # Call LLM with RAG
    print("\nCalling LLM API with RAG...")
    llm_response_with_rag = call_llm_api(prompt)
    
    # Call LLM without RAG
    print("\nCalling LLM API without RAG...")
    llm_response_without_rag = call_llm_api(prompt_without_rag)
    
    # Save with-RAG result
    output_file_with_rag = None
    if llm_response_with_rag:
        print(f"\n[With RAG] Threat Level: {llm_response_with_rag.get('threat_level')}")
        print(f"[With RAG] Priority: {llm_response_with_rag.get('execution_priority')}")
        
        # Save result using reusable function
        try:
            output_file_with_rag = save_action_plan(sample, query, rag_results, llm_response_with_rag, results_action_dir)
            print(f"✓ Saved with-RAG result to {output_file_with_rag.name}")
            print(f"  Full path: {output_file_with_rag}")
        except Exception as e:
            print(f"✗ Error saving with-RAG result: {e}")
            output_file_with_rag = None
    else:
        print("✗ Failed to get LLM response with RAG")
        output_file_with_rag = None
    
    # Save without-RAG result
    without_rag_path = None
    if llm_response_without_rag:
        print(f"\n[Without RAG] Threat Level: {llm_response_without_rag.get('threat_level')}")
        print(f"[Without RAG] Priority: {llm_response_without_rag.get('execution_priority')}")
        
        # Save without-RAG result (pass empty rag_results)
        output_file_without_rag = save_action_plan(sample, query, [], llm_response_without_rag, results_action_dir)
        # Rename to indicate it's without RAG
        without_rag_path = output_file_without_rag.parent / output_file_without_rag.name.replace("action_plan_", "action_plan_noRAG_")
        output_file_without_rag.rename(without_rag_path)
        print(f"✓ Saved without-RAG result to {without_rag_path.name}")
    else:
        print("✗ Failed to get LLM response without RAG")
        output_file_without_rag = None
    
    # Save comparison file if both responses exist
    if llm_response_with_rag and llm_response_without_rag:
        comparison_file = save_comparison_file(
            sample, query, rag_results, 
            llm_response_with_rag, llm_response_without_rag,
            results_action_dir, output_file_with_rag, without_rag_path
        )
        print(f"✓ Saved comparison file to {comparison_file.name}")
    
    print("="*80)

Processing Sample ID: 0
Predicted Label: BENIGN
Confidence: 87.1%

RAG Query: Explain and mitigate a BENIGN attack characterized by elevated UDP traffic patterns.
Found 4 relevant documents

Top 3 documents for LLM context:
  1. What are the key differences between network-layer and application-layer DDoS protection in cloud environments? (Score: 57.64%)
  2. What role does traffic analysis play in mitigating DDoS attacks in cloud environments? (Score: 53.96%)
  3. What are the common challenges faced in defending against DDoS attacks in cloud environments? (Score: 52.48%)

Calling LLM API with RAG...

Calling LLM API without RAG...

[With RAG] Threat Level: Low
[With RAG] Priority: Standard
✓ Saved with-RAG result to action_plan_sample_0_20260208_011438_283081.json
  Full path: RAG_docs\action_plans\action_plan_sample_0_20260208_011438_283081.json

[Without RAG] Threat Level: Low
[Without RAG] Priority: Standard
✓ Saved without-RAG result to action_plan_noRAG_sample_0_20260208_011438_